# Part C Dissertation - Modelling Networks with Complex Weights
## Cyclic SBM for Empirical Graphs
#### _(Candidate 1074092)_

In this notebook, we return to the cyclic SBM to understand whether the spectral gap criterion is suitable for the empirical Motueka and Florida network examples.

Using the spectral clustering algorithm of Gong [9], we have partitions for the empirical graphs. We calculate the key statistics and then substitute their values into the cyclic SBM as below to find $\hat{p}$.

We use the following statistics (see Table 4.3 for more details). <br>
Florida: $17$ in-edges, $11$ out-edges, $p_{in} = 17/41 = 0.414$, $E_{out/in} = 11/17 = 0.65$, derived $p_{out} = 11/82 = 0.134$ <br>
Motueka: $14$ in-edges, $19$ out-edges, $p_{in} = 14/59 = 0.237$, $E_{out/in} = 1.5$, derived $p_{out} = 21/118 = 0.178$

In [1]:
import scipy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import networkx as nx
import scipy.special as sp
import pickle
import os
rng = np.random.default_rng(seed=None)

from numpy.linalg import LinAlgError

In [2]:
def mLapN(M, theta):
    M = M.toarray()
    n = np.shape(M)[0]
    #defines unweighted symmetric matrix
    W_s = (M + M.T) / 2

    #directional info matrix
    A = M
    A[(M > 0) & (M.T == 0)] = 1
    A[(M == 0) & (M.T > 0)] = -1
        
    #T matrix
    T = np.exp(1j * theta * A)
    #L matrix
    L = -W_s * T
    np.fill_diagonal(L, np.sum(W_s, axis=1))

    #Normalisation step, sets L_N = D^(-1/2) * L * D^(-1/2)
    D = np.sum(W_s, axis=1)
    normD = 1 / np.sqrt(D)
    L_N = (normD[:, None] * L) * normD[None, :]
    
    return L_N

def eig_rValuesN(A, rValues):
    sims = len(rValues)
    n = np.shape(A)[0]
    eigArray = np.empty((sims, n + 1), dtype=float)
    #note that we essentially set q = theta/2π, so for varying r as before we have r = 1/q
    thetaValues = (2*np.pi)/np.array(rValues)
    
    for i in range (0, sims):
        L_N = mLapN(A, thetaValues[i])
        eigs = np.linalg.eigvalsh(L_N)
        eigArray[i, 0] = rValues[i]
        for j in range(0, n):
            eigArray[i, j + 1] = eigs[j]
    return eigArray

def eigValues_plot(eigArray, n, idp, graphname):
    plt.figure(figsize=(6, 4))
    for i in range(0, n):
        plt.plot(eigArray[:, 0], eigArray[:, i + 1], label=rf"$\lambda_{{{i + 1}}}$")
    plt.xlabel(f"{idp}")
    plt.ylabel("λ")
    plt.title(f"λ (for varying {idp}) of {graphname}")
    plt.grid(True)
    plt.legend()
    plt.show()

def eigV_rplot(G, graphname):
    A = nx.adjacency_matrix(G)
    rValues = [n for n in range (1, 101)] 
    n = np.shape(G)[0]
    eigArrayN = eig_rValuesN(A, rValues)
    eigValues_plot(eigArrayN, n, "r", graphname)

def eigVsel_plot(eigArray, n, idp, graphname, eigInterest):
    plt.figure(figsize=(6, 4))
    for i in eigInterest:
        plt.plot(eigArray[:, 0], eigArray[:, i], label=rf"$\lambda_{{{i}}}$")
    plt.xlabel(f"{idp}")
    plt.ylabel("λ")
    plt.title(f"λ (for varying {idp}) of {graphname}")
    plt.grid(True)
    plt.legend()
    plt.show()

def eigVsel_rplot(G, graphname, rValues, eigInterest):
    A = nx.adjacency_matrix(G)
    n = np.shape(G)[0]
    eigArrayN = eig_rValuesN(A, rValues)
    eigVsel_plot(eigArrayN, n, "r", graphname, eigInterest)

In [3]:
#alteration to ensure no rows are empty
def sbmCycleGenAlt(clusters, clust_size, p_in, p_out):
    n = clusters * clust_size
    #generates a full matrix of p_out, we will overwrite this later
    A = rng.binomial(1, p_out, size=(n, n))
    #makes diagonal 0 (removes self edges)
    np.fill_diagonal(A, 0)

    #adds the block matrices showing "in" connections from cluster i to cluster i+1
    #we hard code in the bottom corner block
    for i in range(0, clusters-1):
        #creates in blocks
        A_in_block = rng.binomial(1, p_in, size=(clust_size, clust_size))
        #places in blocks above the diagonal
        A[((i)*clust_size):((i+1)*clust_size), ((i+1)*clust_size):((i+2)*clust_size)] = A_in_block
        #adds extra entries if we have a zero row
        for j in range(i * clust_size, i * (clust_size + 1)):
            if np.all(A[j] == 0) == True:
                rand1 = np.random.randint(i * clust_size, i * (clust_size + 1))
                rand2 = np.random.randint(0, n)
                A[j, rand1] = 0.1
                A[j, rand2] = 0.1
    A_in_block = rng.binomial(1, p_in, size=(clust_size, clust_size))
    A[(n-clust_size):n, 0:clust_size] = A_in_block
    #adds extra entries if we have a zero row
    for j in range(n - clust_size, n):
        if np.all(A[j] == 0) == True:
            rand1 = np.random.randint(0, clust_size)
            rand2 = np.random.randint(0, n)
            A[j, rand1] = 0.1
            A[j, rand2] = 0.1
    return A

In [4]:
def sbmClusterTest_p_outAlt(clusters, clust_size, p_inValues, p_outValues, sims, rValues):
    #rValues = [2, 3, 4, 5, 6] #determines the number of clusters we test for, add more if needed
    rValLength = len(rValues)
    full_results = {}
    results = {}
    errors = 0
    for p_in in p_inValues:
        full_results[p_in] = {}
        results[p_in] = []
        for p_out in p_outValues:
            full_results[p_in][p_out] = np.zeros(sims) 

    for p_in in p_inValues:
        for p_out in p_outValues:
            for i in range(0, sims):
                A = sbmCycleGenAlt(clusters, clust_size, p_in, p_out)
                G = nx.from_numpy_array(A, create_using=nx.DiGraph)
                A = nx.adjacency_matrix(G)
                try:
                    eigArray = eig_rValuesN(A, rValues)
                    spectralGap = np.zeros(rValLength)
                    for k in range(0, rValLength):
                        spectralGap[k] = eigArray[k, 2] - eigArray[k, 1]
                    rChoice = rValues[np.argmax(spectralGap)]
                    if rChoice == clusters:
                        full_results[p_in][p_out][i] = 1
                except LinAlgError as e:
                    errors += 1
            results[p_in].append(np.sum(full_results[p_in][p_out])/(sims - errors))

    return results, full_results

In [5]:
#Florida and Motueka test
#Florida: 17 in, 11 out, p_in = 17/41 = 0.414, E_out/in = 11/17 = 0.65, derived p_out = 11/82 = 0.134
clusters = 3
clust_size = 4
p_inValues = [17/41]
p_outValues = [11/82]
sims = 1000
rValues = [3, 4, 5, 6]
results, full_results = sbmClusterTest_p_outAlt(clusters, clust_size, p_inValues, p_outValues, sims, rValues)

C:\Users\rapha\AppData\Local\Temp\ipykernel_13960\3720646719.py:20: RuntimeWarning: divide by zero encountered in divide
  normD = 1 / np.sqrt(D)
C:\Users\rapha\AppData\Local\Temp\ipykernel_13960\3720646719.py:21: RuntimeWarning: invalid value encountered in multiply
  L_N = (normD[:, None] * L) * normD[None, :]


In [6]:
print(results)

{0.4146341463414634: [0.32653061224489793]}


In [7]:
#Motueka: 14 in, 19 out, p_in = 14/59 = 0.237, E_out/in = 1.5, derived p_out = 21/118 = 0.178
clusters = 3
clust_size = 5
p_inValues = [14/59]
p_outValues = [21/118]
sims = 1000
rValues = [3, 4, 5, 6]
results, full_results = sbmClusterTest_p_outAlt(clusters, clust_size, p_inValues, p_outValues, sims, rValues)

C:\Users\rapha\AppData\Local\Temp\ipykernel_13960\3720646719.py:20: RuntimeWarning: divide by zero encountered in divide
  normD = 1 / np.sqrt(D)
C:\Users\rapha\AppData\Local\Temp\ipykernel_13960\3720646719.py:21: RuntimeWarning: invalid value encountered in multiply
  L_N = (normD[:, None] * L) * normD[None, :]


In [8]:
print(results)

{0.23728813559322035: [0.21658031088082902]}


For both the Florida and Motueka networks, we find that $\hat{p}$ is abysmally low. This is to be expected, as $\hat{p}$ drops below reliable thresholds for $r=3$ at around $E_{out/in} \approx 0.34$ so the amount of noise in the Florida and Motueka networks is too high for the spectral gap criterion to be reliable.